# Impact des athlètes de haut niveau et des performances françaises sur les licences sportives

Dans ce projet, on cherche à voir si les grandes performances françaises (titres mondiaux, médailles olympiques, émergence de stars médiatiques) provoquent vraiment une vague d'inscriptions en club. Au-delà du simple nombre de médailles, on essaie surtout de comprendre par quels mécanismes ça passe, que ce soit la proximité géographique avec un champion, l'identification par l'âge ou le genre, ou encore le coût d'accès à la pratique.

Pour croiser ces différents angles, on s'appuie sur 4 sources principales :
- les données de recensement des licences de l'INJEP, avec la base départementale 2016-2023 complétée par 2025, et l'historique national depuis 2002
- les résultats des athlètes français aux JO de Paris 2024 (data.gouv.fr)
- les vues Wikipédia via l'API Wikimedia, utilisées comme proxy de l'attention médiatique autour des sportifs
- les activations du Pass'Sport (campagnes 2022 à 2024) pour regarder ce qui se passe chez les jeunes de foyers modestes

In [1]:
# à décommenter si les packages ne sont pas encore installés
%pip install -q -r requirements.txt

from pathlib import Path
import pandas as pd
import plotly.express as px

import src.collect_licences as cl
import src.collect_jo as cjo
import src.collect_trends as ct
import src.collect_passsport as cp
import src.processing as pr
import src.viz as vz

pd.set_option("display.float_format", "{:,.2f}".format)

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# chargement des bases (on passe par les parquets/csv si deja generes)
chemin_complet = Path("data/processed/licences_dep_2016_2025.parquet")
if chemin_complet.exists():
    df_licences = pd.read_parquet(chemin_complet)
else:
    chemin_hist = Path("data/processed/licences_dep.parquet")
    df_hist = pd.read_parquet(chemin_hist) if chemin_hist.exists() else cl.construire_base_departementale()
    df_licences = cl.enrichir_avec_2025(df_hist)

df_histo = cl.charger_historique_national()
df_athletes_2024 = cjo.charger_donnees_jo()["athletes_paris_2024"]
df_pass_dep, df_pass_hebdo = cp.construire_bases_passsport(supprimer_csv_bruts=False)

# codes injep des federations qu'on suit dans le rapport
CODES_CLES = {
    "FFTT": "124",
    "FFC": "108",
    "FFJDA": "117",
    "FFN": "119",
    "FFE": "110",
    "FFF": "111",
}

print("Licences dep :", len(df_licences), "| annees :", sorted(df_licences["annee"].unique().tolist()))
print("Historique :", len(df_histo), "| Pass'Sport :", len(df_pass_dep))

/home/onyxia/work/Projet-Python-2AS1/src/collect_passsport.py:183: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dates = pd.to_datetime(chunk[mapping["date"]], dayfirst=True, errors="coerce")
/home/onyxia/work/Projet-Python-2AS1/src/collect_passsport.py:183: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dates = pd.to_datetime(chunk[mapping["date"]], dayfirst=True, errors="coerce")
/home/onyxia/work/Projet-Python-2AS1/src/collect_passsport.py:183: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dates = pd.to_datetime(chunk[mapping["date"]], dayfirst=True, errors="coerce")
/home/onyxia/work/Projet-Python-2AS1/src/collect_passsport.py:183: UserWarning: Parsing dates in %Y-%m-%d format wh

Licences dep : 117233 | annees : [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2025]
Historique : 3144 | Pass'Sport : 36371
NaN dans df_licences : 0


/home/onyxia/work/Projet-Python-2AS1/src/collect_passsport.py:183: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dates = pd.to_datetime(chunk[mapping["date"]], dayfirst=True, errors="coerce")


In [3]:
#On cherche les valeurs manquantes sur toutes les tables que l'on vient de créer
tables_a_auditer = {
    "df_licences": df_licences,
    "df_histo": df_histo,
    "df_athletes_2024": df_athletes_2024,
    "df_pass_dep": df_pass_dep,
    "df_pass_hebdo": df_pass_hebdo,
}

for nom_df, d in tables_a_auditer.items():
    nb_na = int(d.isna().sum().sum())
    cols_na = d.isna().sum()
    cols_na = cols_na[cols_na > 0].to_dict()
    nb_doublons = int(d.duplicated().sum())
    print(f"[{nom_df}] lignes={len(d):,} | NaN total={nb_na} {cols_na if cols_na else ''} | doublons exacts={nb_doublons}")

[df_licences] lignes=117,233 | NaN total=0  | doublons exacts=0
[df_histo] lignes=3,144 | NaN total=402 {'licences': 402} | doublons exacts=0
[df_athletes_2024] lignes=11,110 | NaN total=10501 {'Taille (en cm)': 6033, 'Site de compétition': 346, 'Latitude': 1715, 'Longitude': 1715, 'Date de début': 346, 'Date de fin': 346} | doublons exacts=0
[df_pass_dep] lignes=36,371 | NaN total=28397 {'departement': 322, 'code_fed': 28075} | doublons exacts=0
[df_pass_hebdo] lignes=7,217 | NaN total=5891 {'code_fed': 5891} | doublons exacts=0


## 1. Vue d'ensemble : est-ce que les grands événements font vraiment exploser les licences ?

### 1.1. Dispersion territoriale et trajectoire sur le temps long (2002-2025)

Avant de rentrer dans des cas précis, on commence par regarder comment se comporte la croissance entre 2023 et 2025 d'un département à l'autre sur nos disciplines principales, puis on replace les chiffres récents dans la série longue depuis 2002 pour voir si Paris 2024 sort vraiment du lot par rapport aux olympiades précédentes.

In [3]:
# dispersion des croissances par departement (2023-2025)
feds_dispersion = {
    "Tennis de table": CODES_CLES["FFTT"],
    "Escrime": CODES_CLES["FFE"],
    "Judo": CODES_CLES["FFJDA"],
    "Football": CODES_CLES["FFF"],
    "Cyclisme": CODES_CLES["FFC"],
    "Natation": CODES_CLES["FFN"],
}
vz.plot_boxplot_dispersion_departementale(df_licences, feds_dispersion, 2023, 2025).show()

In [4]:
# total national depuis 2002
df_total_fr = df_histo.groupby("annee", as_index=False)["licences"].sum().sort_values("annee")

fig_fr = px.line(
    df_total_fr,
    x="annee",
    y="licences",
    markers=True,
    title="Licenciés sportifs en France (2002-2025)",
    labels={"annee": "Année", "licences": "Licenciés"},
)
fig_fr.update_traces(line_width=3)
for annee_jo, ville in vz.OLYMPIADES:
    fig_fr.add_vline(
        x=annee_jo,
        line_dash="dash",
        line_color="#d62728" if annee_jo == 2024 else "gray",
        annotation_text=ville,
    )
fig_fr.show()

vz.plot_historique_olympique(df_histo, CODES_CLES["FFTT"], "Tennis de table").show()

Sur les boxplots, on voit tout de suite que la hausse du tennis de table et de l'escrime n'est pas tirée par deux ou trois départements isolés, la quasi-totalité du territoire est dans le positif avec une médiane autour de +25 % pour le ping-pong. À l'inverse, pour le cyclisme et la natation, la médiane départementale reste collée autour de 0 %

Si on prend un peu de recul depuis 2002, le mouvement sportif global progresse doucement sur la durée, avec un gros creux évident en 2020-2021 pendant le Covid. Le cas de la FFTT est beaucoup plus frappant, le nombre de licenciés stagnait sous les 195 000 depuis vingt ans sans jamais décoller après Athènes, Pékin, Londres ou Rio, puis d'un coup en 2025 la fédération passe au-dessus des 222 000 licenciés

### 1.2. Médailles à Paris 2024 et croissance des licences (2023-2025)

On croise maintenant le nombre de médailles obtenues par les Français à Paris 2024 avec l'évolution des licenciés entre 2023 (dernière saison normale avant les Jeux) et 2025.

In [5]:
df_fr_2024 = pr.synthetiser_paris_2024_france(df_athletes_2024)
df_bilan_2024 = pr.croiser_medailles_et_croissance(df_licences, df_fr_2024, 2023, 2025)

display(df_bilan_2024[["nom_fed", "disciplines_jo", "total_medailles", "medailles_or", "croissance_pct", "gain_net"]])
vz.plot_medailles_vs_croissance(df_bilan_2024).show()

,nom_fed,disciplines_jo,total_medailles,medailles_or,croissance_pct,gain_net
19,Fédération Française de Tennis de Table,Tennis de table,4,0,25.28,"44,849.00"
8,Fédération Française d'Escrime,Escrime,16,1,21.85,"11,702.00"
2,Fédération Française de Badminton,Badminton,0,0,18.72,"37,789.00"
15,Fédération Française de Lutte,Lutte,0,0,16.78,"4,755.00"
22,Fédération Française de Triathlon et Disciplin...,Triathlon,2,1,16.22,"9,903.00"
21,Fédération Française de Tir à l'Arc,Tir à l'arc,4,0,14.96,"11,049.00"
11,"Fédération Française d'Haltérophilie, Musculation",Haltérophilie,0,0,14.04,"2,929.00"
20,Fédération Française de Tir,Tir,1,0,13.31,"34,064.00"
12,Fédération Française de Handball,Handball,17,0,13.23,"70,353.00"
24,Fédération Française de Volley,"Volleyball, Volleyball de plage",12,12,12.18,"17,999.00"


Clairement, gagner beaucoup de médailles ne garantit pas une explosion des inscriptions, il n'y a pas de lien linéaire sur le nuage de points.

Le tennis de table arrive tout en haut avec +25,3 % (+44 849 licenciés) alors qu'il ne compte que 4 médailles individuelles cumulées, juste devant l'escrime (+21,8 % pour 16 médailles, très exposée au Grand Palais). Le vrai paradoxe, c'est la natation : malgré les 4 titres de Léon Marchand et 12 médailles au total, les effectifs restent quasiment plats à +0,2 %. C'est probablement un problème très concret de capacité d'accueil, les piscines municipales sont déjà saturées et les clubs ne peuvent pas absorber plus de monde.

De l'autre côté, des sports plus chers ou plus contraignants niveau matériel reculent malgré des podiums, comme l'équitation (-7,4 %) ou le canoë-kayak (-10,3 %). On remarque aussi que le badminton (+18,7 %) ou la lutte (+16,8 %) progressent fort sans aucune médaille, ce qui rappelle que la tendance de fond compte aussi beaucoup.

### 1.3. Point de comparaison hors JO : la Coupe du Monde de Football 2018

Comme notre sujet porte sur les performances de la France en général et pas uniquement sur Paris 2024, on a voulu faire un parallèle avec la victoire des Bleus en 2018 sur la Fédération Française de Football (code `111`). On compare l'évolution à Paris face au reste du pays en base 100 sur 2016.

In [6]:
# base 100 en 2016 sur le foot (code 111)
df_foot_100 = pr.preparer_serie_base_100(
    df_licences, CODES_CLES["FFF"], dep_cible="75", annee_ref=2016, label_cible="Paris (75)", col_fed="code_fed"
)
ecart_foot = pr.calculer_ecart_double_difference(df_foot_100, "Paris (75)", 2016, 2023)
print(f"ecart Paris vs reste de la France (2016-2023) : {ecart_foot:+.1f} pts")

vz.plot_effet_local_base100(
    df_foot_100, "Football", "Coupe du Monde 2018", annee_ref=2016, annee_evenement=2018.5, evenement_label="CDM 2018"
).show()

ecart Paris vs reste de la France (2016-2023) : +9.0 pts


On observe bien un petit coup d'accélérateur juste après juillet 2018, entre la saison 2018 et 2019, avant que le Covid ne coupe l'élan en 2020 et 2021. Sur l'ensemble de la période 2016-2023, Paris garde une avance de +9 points d'indice par rapport au reste de la France, ce qui peut s'expliquer par le dynamisme démographique urbain mais aussi par l'aura des joueurs franciliens de l'équipe de France comme Kylian Mbappé. Par contre, comme le foot part déjà d'une base énorme, le saut en pourcentage reste bien plus modeste que ce qu'on voit sur le ping-pong.

## 2. Zoom sur le cas des frères Lebrun (FFTT)

### 2.1. Effet local : l'Hérault (34) face au reste de la France

Pour tester s'il existe un effet "héros local" dans le département d'origine d'Alexis et Félix Lebrun, on fait une double différence entre l'Hérault et le reste de la France avec deux années de référence :
- base 100 en 2019 pour capter tout l'effet depuis avant le Covid, quand ils commencent à se faire connaître
- base 100 en 2023 pour isoler uniquement le choc des JO de Paris 2024

In [7]:
# base 2019 : effet global depuis l'avant-covid
df_ping_2019 = pr.preparer_serie_base_100(
    df_licences, CODES_CLES["FFTT"], "34", annee_ref=2019, label_cible="Hérault (34)", col_fed="code_fed"
)
print(f"ecart Herault 2019-2025 : {pr.calculer_ecart_double_difference(df_ping_2019, 'Hérault (34)', 2019, 2025):+.1f} pts")
vz.plot_effet_local_base100(df_ping_2019, "Tennis de table", "Frères Lebrun", annee_ref=2019).show()

# base 2023 : juste le choc post-JO
df_ping_2023 = pr.preparer_serie_base_100(
    df_licences, CODES_CLES["FFTT"], "34", annee_ref=2023, label_cible="Hérault (34)", col_fed="code_fed"
)
print(f"ecart Herault 2023-2025 : {pr.calculer_ecart_double_difference(df_ping_2023, 'Hérault (34)', 2023, 2025):+.1f} pts")
vz.plot_effet_local_base100(df_ping_2023, "Tennis de table", "Frères Lebrun", annee_ref=2023).show()

df_ping_carte = pr.calculer_croissance_departementale(df_licences, CODES_CLES["FFTT"], 2023, 2025, col_fed="code_fed")
vz.plot_carte_croissance_dep(df_ping_carte, "Tennis de table", 2023, 2025).show()

ecart Herault 2019-2025 : +15.1 pts


ecart Herault 2023-2025 : -0.3 pts


La comparaison des deux bases est assez parlante. En prenant 2019 comme point de départ, l'Hérault finit avec +15,1 points d'indice d'avance sur le reste de la France en 2025, et on voit que le décrochage commence déjà entre 2021 et 2023, au moment de leurs premiers titres nationaux et européens.

En revanche, quand on recale tout en base 100 en 2023, l'écart disparaît complètement (-0,3 pt) et les deux courbes se collent l'une à l'autre. Autrement dit, les frères Lebrun avaient déjà créé une avance locale avant 2024, puis les Jeux ont diffusé l'engouement partout en France de façon très homogène, ce que confirme la carte départementale presque entièrement en bleu.

### 2.2. Qui sont les nouveaux licenciés ? Regard sur l'âge et le genre

On regarde maintenant si la vague d'adhésions à la FFTT concerne surtout les jeunes garçons, qui sont a priori le public le plus proche du profil des frères Lebrun.

In [8]:
# repartition par age (2019-2025)
df_ping_ages = cl.construire_base_ages_2019_2025(CODES_CLES["FFTT"])
piv_age = df_ping_ages.pivot(index="tranche_age", columns="annee", values="licences")
piv_age["gain_2023_2025"] = piv_age[2025] - piv_age[2023]
piv_age["part_du_gain (%)"] = piv_age["gain_2023_2025"] / piv_age["gain_2023_2025"].sum() * 100
display(piv_age)
vz.plot_effet_age(df_ping_ages, "Tennis de table", "Frères Lebrun", annee_ref=2023).show()

# evolution femmes vs hommes
df_ping_genre = pr.analyser_effet_genre(df_licences, CODES_CLES["FFTT"], annee_ref=2019)
display(df_ping_genre[["annee", "licences_f", "licences_h", "taux_feminisation", "indice_f", "indice_h"]])
vz.plot_effet_genre(df_ping_genre, "Tennis de table", "Frères Lebrun", annee_ref=2019).show()

annee,2019,2020,2021,2022,2023,2025,gain_2023_2025,part_du_gain (%)
tranche_age,,,,,,,,
10 à 14 ans,"55,959.00","47,864.00","32,160.00","41,470.00","47,406.00","60,259.00","12,853.00",28.66
15 à 19 ans,"15,905.00","13,333.00","10,987.00","12,345.00","12,838.00","14,250.00","1,412.00",3.15
20 ans et plus,"97,561.00","91,816.00","70,603.00","84,583.00","95,196.00","121,262.00","26,066.00",58.12
Moins de 10 ans,"41,928.00","22,792.00","12,318.00","19,715.00","21,973.00","26,491.00","4,518.00",10.07


,annee,licences_f,licences_h,taux_feminisation,indice_f,indice_h
0,2016,38767,166897,18.85,92.50,98.43
1,2017,38358,165983,18.77,91.53,97.89
2,2018,40199,166444,19.45,95.92,98.16
3,2019,41909,169566,19.82,100.00,100.00
4,2020,24421,151489,13.88,58.27,89.34
5,2021,16436,109742,13.03,39.22,64.72
6,2022,23722,134391,15.00,56.60,79.26
7,2023,26953,150460,15.19,64.31,88.73
8,2025,32364,189898,14.56,77.22,111.99


Sur l'âge, le résultat est un peu moins caricatural qu'on aurait pu le penser. Entre 2023 et 2025, les 10-14 ans progressent très vite (+27,1 %, soit près de 29 % des nouveaux arrivants), ce qui colle bien à l'effet d'identification chez les collégiens. Mais les 20 ans et plus progressent exactement au même rythme (+27,4 %) et représentent à eux seuls 58 % du gain total (+26 066 licences), probablement des jeunes adultes ou d'anciens joueurs qui reprennent en club. Les moins de 10 ans rebondissent aussi (+20,6 %), alors que les 15-19 ans bougent moins (+11 %), souvent pris par le lycée et les études.

Côté genre, l'écart est beaucoup plus net. En base 2019, les hommes montent à l'indice 112 en 2025 (+20 332 licences par rapport à 2019), tandis que les femmes restent à 77,2 (-9 545 licences) avec un taux de féminisation tombé à 14,6 %. Il faut quand même nuancer la grosse chute féminine de 2020, qui vient en partie d'un changement dans le comptage des licences scolaires et loisirs dans les stats INJEP, mais même après 2023 le rattrapage reste nettement plus masculin.

## 3. Comment fonctionne l'effet "role model" ?

### 3.1. L'attention médiatique mesurée par Wikipédia

Pour relier les exploits sportifs aux inscriptions de septembre 2024, on utilise les consultations de pages Wikipédia de quelques figures fortes des Jeux, d'abord par mois sur 2023-2025 puis au jour le jour pendant la quinzaine olympique.

In [9]:
panel_athletes = {
    "Félix Lebrun": "Félix_Lebrun",
    "Alexis Lebrun": "Alexis_Lebrun",
    "Pauline Ferrand-Prévot": "Pauline_Ferrand-Prévot",
    "Teddy Riner": "Teddy_Riner",
    "Léon Marchand": "Léon_Marchand",
}

df_wiki_mois = ct.recuperer_vues_wikipedia(panel_athletes, "20230101", "20250301", granularite="monthly")
moy_2023 = df_wiki_mois[df_wiki_mois["date"].dt.year == 2023].groupby("personnalite")["vues"].mean()
pic_2024 = df_wiki_mois[df_wiki_mois["date"].dt.year == 2024].groupby("personnalite")["vues"].max()
display(pd.DataFrame({"moy_2023": moy_2023.round(0), "pic_2024": pic_2024, "ratio": (pic_2024 / moy_2023).round(1)}))

px.line(df_wiki_mois, x="date", y="vues", color="personnalite", markers=True, title="Vues Wikipédia mensuelles (2023-2025)").show()

# zoom pendant les JO
df_wiki_jour = ct.recuperer_vues_wikipedia(panel_athletes, "20240720", "20240820", granularite="daily")
px.line(df_wiki_jour, x="date", y="vues", color="personnalite", title="Vues Wikipédia journalières pendant les JO 2024").show()

,moy_2023,pic_2024,ratio
personnalite,,,
Alexis Lebrun,"10,450.00",181592,17.40
Félix Lebrun,"11,998.00",286477,23.90
Teddy Riner,"35,840.00",1128289,31.50


Les courbes journalières montrent bien deux profils d'exposition différents pendant l'été 2024. Des athlètes comme Pauline Ferrand-Prévot ou Cassandre Beaugrand font un pic énorme le jour de leur médaille d'or puis l'attention retombe très vite, alors que les frères Lebrun (tout comme Léon Marchand) restent dans l'actualité sur presque toute la durée des Jeux grâce à l'enchaînement des tours en simple et par équipes. Sur l'année, les vues mensuelles sont multipliées par plus de 23 pour Félix Lebrun et 31 pour Teddy Riner au moment du pic d'août.

### 3.2. Identification genrée et barrières à l'entrée selon les sports

Pour vérifier si le genre du champion influence directement le profil des nouveaux inscrits, on compare la croissance des licences femmes et hommes entre 2023 et 2025 sur cinq disciplines associées à des têtes d'affiche bien identifiées.

In [10]:
sports_role_models = {
    "Tennis de table (Frères Lebrun)": CODES_CLES["FFTT"],
    "Escrime (M. Apithy-Brunet / Escrimeurs)": CODES_CLES["FFE"],
    "Judo (C. Agbegnenou / T. Riner)": CODES_CLES["FFJDA"],
    "Cyclisme (P. Ferrand-Prévot)": CODES_CLES["FFC"],
    "Natation (L. Marchand)": CODES_CLES["FFN"],
}

vz.plot_dumbbell_genre_role_models(df_licences, sports_role_models, 2023, 2025).show()

Ce graphique résume assez bien le mécanisme d'identification genrée. Le tennis de table est la seule des cinq disciplines où la hausse masculine (+26,2 %) dépasse nettement la hausse féminine (+20,1 %). Partout ailleurs, quand une championne française a marqué les Jeux, ce sont les femmes qui tirent la croissance : +32,1 % contre +17,0 % en escrime avec le titre de Manon Apithy-Brunet, +17,8 % contre +9,4 % au judo où Clarisse Agbegnenou et Romane Dicko partagent l'affiche avec Teddy Riner, et +6,1 % contre +1,4 % en cyclisme après le sacre de Pauline Ferrand-Prévot.

En même temps, le contraste entre le ping-pong et le cyclisme montre les limites de la seule visibilité médiatique. Même avec un pic d'attention très fort pour Pauline Ferrand-Prévot, le coût d'un VTT ou d'un vélo de route reste un vrai frein pour franchir le pas en club, là où une raquette de tennis de table est accessible presque immédiatement.

### 3.3. Révélation médiatique contre icône installée : Frères Lebrun vs Teddy Riner

Un autre facteur possible, c'est l'effet de nouveauté. Teddy Riner est au sommet du judo français depuis 2008, alors que les frères Lebrun arrivent comme une révélation pour le grand public. On compare leurs deux fédérations sur le temps long (depuis Pékin 2008) et on regarde ce que donnent les rebonds à t+1 et t+2 après chaque olympiade.

In [11]:
feds_riner_lebrun = {
    "Tennis de table (Frères Lebrun)": CODES_CLES["FFTT"],
    "Judo (Teddy Riner)": CODES_CLES["FFJDA"],
}

df_comp_judo = pr.comparer_deux_federations_base100(df_licences, feds_riner_lebrun, annee_ref=2023)
vz.plot_comparaison_federations_base100(
    df_comp_judo, annee_ref=2023, titre="Tennis de table (FFTT) vs Judo (FFJDA)"
).show()

vz.plot_comparaison_longue_base100(df_histo, feds_riner_lebrun, annee_ref=2008).show()

df_persistance = pr.analyser_persistance_olympique(
    df_histo,
    {"Tennis de table": CODES_CLES["FFTT"], "Cyclisme": CODES_CLES["FFC"], "Judo": CODES_CLES["FFJDA"]},
)
display(df_persistance)

,sport,olympiade,licences_jo,evol_t1_pct,evol_t2_pct
0,Tennis de table,2008,"180,195.00",5.38,6.43
1,Tennis de table,2012,"190,539.00",1.42,-0.57
2,Tennis de table,2016,"180,574.00",-0.72,-2.22
3,Tennis de table,2024,"182,499.00",21.79,NaN
4,Cyclisme,2008,"103,070.00",1.76,5.33
5,Cyclisme,2012,"115,329.00",3.40,3.07
6,Cyclisme,2016,"118,611.00",-1.54,-2.14
7,Cyclisme,2024,"107,508.00",2.15,NaN
8,Judo,2008,"553,391.00",3.76,4.86
9,Judo,2012,"573,457.00",0.60,-0.98


Le judo gagne beaucoup de monde en volume après Paris 2024 (+64 354 licenciés), mais en proportion la hausse reste deux fois moindre que celle du tennis de table (+12,2 % contre +25,3 %). Comme Teddy Riner est déjà une figure connue depuis plus de quinze ans et que le judo part d'un socle déjà massif, son sacre joue plutôt un rôle de confirmation et de fidélisation, là où les frères Lebrun créent un vrai choc de découverte sur une fédération qui stagnait depuis 2008.

## 4. Est-ce que les publics modestes suivent la tendance ? Analyse du Pass'Sport

Pour finir, on regarde si cet engouement touche aussi les jeunes issus de foyers plus modestes grâce aux données du Pass'Sport (l'aide de 50 euros pour prendre une licence). Petit point de calendrier à avoir en tête : la campagne Pass'Sport 2023 correspond à la saison 2023-2024 (avant les Jeux), et la campagne 2024 finance la rentrée 2024-2025, donc celle qui s'aligne sur le millésime 2025 de l'INJEP.

In [12]:
feds_passsport = {
    "Tennis de table (FFTT)": CODES_CLES["FFTT"],
    "Escrime (FFE)": CODES_CLES["FFE"],
    "Judo (FFJDA)": CODES_CLES["FFJDA"],
    "Cyclisme (FFC)": CODES_CLES["FFC"],
    "Natation (FFN)": CODES_CLES["FFN"],
}

# evolution annuelle des activations par discipline
piv_pass = (
    df_pass_dep[df_pass_dep["code_fed"].isin(feds_passsport.values())]
    .pivot_table(index="code_fed", columns="campagne", values="nb_passsport", aggfunc="sum", fill_value=0)
)
piv_pass.index = piv_pass.index.map({v: k for k, v in feds_passsport.items()})
if 2023 in piv_pass.columns and 2024 in piv_pass.columns:
    piv_pass["evol_2023_2024 (%)"] = (piv_pass[2024] / piv_pass[2023] - 1) * 100
display(piv_pass.sort_values("evol_2023_2024 (%)", ascending=False))

vz.plot_evolution_passsport_sports(df_pass_dep, feds_passsport, campagne_ref=2023).show()

# croisement genre x pass'sport sur ping-pong et cyclisme
pass_genre = (
    df_pass_dep[
        df_pass_dep["code_fed"].isin([CODES_CLES["FFTT"], CODES_CLES["FFC"]])
        & df_pass_dep["campagne"].isin([2023, 2024])
    ]
    .pivot_table(index=["code_fed", "genre"], columns="campagne", values="nb_passsport", aggfunc="sum", fill_value=0)
)
if 2023 in pass_genre.columns and 2024 in pass_genre.columns:
    pass_genre["evol_2023_2024 (%)"] = (pass_genre[2024] / pass_genre[2023] - 1) * 100
display(pass_genre)

campagne,2022,2023,2024,evol_2023_2024 (%)
code_fed,,,,
Tennis de table (FFTT),15363,15505,22517,45.22
Judo (FFJDA),92599,100196,119210,18.98
Natation (FFN),46498,47332,55505,17.27
Cyclisme (FFC),10292,9665,11024,14.06


campagne         2023   2024  evol_2023_2024 (%)
code_fed genre                                  
108      F       1164   1323               13.66
         H       8501   9701               14.12
124      F       2588   3430               32.53
         H      12917  19087               47.77

Les chiffres du Pass'Sport confirment plutôt bien ce qu'on a vu sur l'ensemble des licences. À la rentrée 2024, les activations augmentent nettement sur les disciplines qui ont brillé aux Jeux, en particulier sur le tennis de table et l'escrime. L'aide de 50 euros joue bien son rôle d'amortisseur dans un sport où le coût de la licence et de l'équipement reste raisonnable, alors que l'effet est beaucoup plus limité sur des sports plus chers comme le cyclisme. 

Le croisement par genre va aussi dans le même sens que le reste du rapport, avec une dynamique plus marquée chez les garçons en tennis de table et un meilleur équilibre relatif sur le cyclisme.

## 5. Conclusion et limites de l'analyse

Au final, nos données montrent que les performances des athlètes français ne déclenchent pas automatiquement un boom des licences. Pour que l'engouement médiatique se transforme vraiment en inscriptions en club, plusieurs conditions doivent être réunies en même temps :
- un sport accessible matériellement et avec de la place dans les clubs, ce qui explique l'écart énorme entre le tennis de table (+25,3 %) et la natation (+0,2 %) ou le cyclisme (+2,0 %)
- un effet d'incarnation fort et visible sur plusieurs jours, qui passe à la fois par l'ancrage territorial (l'Hérault avait déjà pris +15 points d'avance avant les Jeux) et par l'identification genrée (les femmes progressent plus vite dans les sports portés par des championnes, les hommes tirent le ping-pong)
- un effet de nouveauté, plus fort en proportion pour une révélation comme les frères Lebrun que pour une icône déjà installée comme Teddy Riner

Il reste quand même quelques limites à garder en tête dans l'interprétation :
- les séries INJEP ont connu une rupture en 2020-2021 qui mélange crise sanitaire et évolution du comptage des licences scolaires, d'où le fait de travailler à la fois sur base 2019 et base 2023
- sur le Pass'Sport, une partie de la hausse peut venir d'un effet d'aubaine (des familles déjà licenciées qui pensent davantage à activer leur aide), même si la comparaison entre fédérations permet d'isoler ce qui dépasse la moyenne
- enfin, avec le millésime 2025, on ne mesure que l'afflux immédiat de la rentrée post-JO, il faudra attendre les saisons 2026 et 2027 pour savoir combien de ces nouveaux licenciés restent vraiment en club sur la durée